In [1]:
import os
from os.path import exists
import glob
import json
import pdb
import numpy as np
from skimage import draw
import matplotlib.pyplot as plt
import argparse
import pyfiglet
from skimage import measure
from tqdm import tqdm
from PIL import Image
import pyvips as Vips
# import openslide
from Reinhard import Reinhard


In [2]:
ID_MASK_SHAPE = (1024, 1024)

# Color Coding
lablel2id = {'True':'50', 'Pre':'100',
             'False':'150', 'Unknown':'0'}

DATASET_PATH = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/generated_data_v2/"
DATASET_PATH = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/generated_data_v2/Unnormalized/"

In [3]:
DATASET_PATH = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/patient_wise_crops_LB/"

In [4]:

def normalization(REF_IMG_PATH):
    print("Init Normalization")
    ref_image = Vips.Image.new_from_file(REF_IMG_PATH)
    normalizer = Reinhard()
    normalizer.fit(ref_image)
    return normalizer


def save_img(img, file_name, tileX, tileY, save_dir, label="mask"):
    im = Image.fromarray(img)

    file_name = file_name + "_" + str(tileX)+"x" + "_" + str(tileY) + "y" + "_" + label + ".png"

    save_name = os.path.join(save_dir, file_name)
    im.save(save_name)


def polygon2id(image_shape, mask, ids, coords_x, coords_y):
    vertex_row_coords, vertex_col_coords = coords_y, coords_x
    fill_row_coords, fill_col_coords = draw.polygon(
        vertex_row_coords, vertex_col_coords, image_shape)

    # Row and col are flipped
    mask[fill_col_coords, fill_row_coords] = ids
    return mask

def polygon2mask1(image_shape, mask, color, coords_x, coords_y):
    """Compute a mask with labels having different colors
    from polygon.
    Parameters
    ----------
    image_shape : tuple of size 2.
        The shape of the mask.
    coords_x: X coordinates
    coords_y: Y coordinates
    mask : Mask with same size of the image (initially empty
    mask is given as input)
    Returns
    -------
    mask : 2-D ndarray of type 'bool'.
        The mask that corresponds to the input polygon.
    """

    vertex_row_coords, vertex_col_coords = coords_x, coords_y
    fill_row_coords, fill_col_coords = draw.polygon(vertex_row_coords, vertex_col_coords, image_shape)

    # Row and col are flipped
    mask[fill_col_coords, fill_row_coords] = color

    # mask[fill_row_coords, fill_col_coords] = color
    return mask

In [5]:
def get_vips_info(vips_img):
    # # Get bounds-x and bounds-y offeset
    #print(vips_img.get_fields())
    vfields = [f.split('.') for f in vips_img.get_fields()]
    #print("--------------",vfields)
    #vfields = [f for f in vfields if f[0] == 'openslide']
    vfields = [f for f in vfields]
    vfields = dict([('.'.join(k[1:]), vips_img.get('.'.join(k))) for k in vfields])
    print(vfields)
    return vfields

In [6]:
def process_json(WSI_path, json_path,  visualize=False):
    """This function is used to read and process the json files
    and generate save generated masks

    Parameters
    -----------
    json_path : path to json file
    save_dir : dir where the generated masks will be saved
    visualize : True , if you want to see the mask generated
    """


    # Mask Folder
    mask_save_dir = os.path.join(DATASET_PATH, "labels")
    if not os.path.exists(mask_save_dir):
        os.makedirs(mask_save_dir)

    # Image Folder
    image_save_dir = os.path.join(DATASET_PATH, "images")
    if not os.path.exists(image_save_dir):
        os.makedirs(image_save_dir)


    imagenames = glob.glob(os.path.join(WSI_path, "*.svs"))
    imagenames = sorted(imagenames)
    
    plaque_dict = {'True': 0, 'Pre': 0, 'False': 0,'Unknown': 0}

    for img in imagenames:
        # Read the WSI image
        vips_img = Vips.Image.new_from_file(img, level=0)
        vinfo = get_vips_info(vips_img)
        # Get the corresponding json file
        # json_file_name = os.path.basename(img).split(".svs")[0] + ".json"
        json_file_name = os.path.basename(img) + ".json"
        json_file_name = os.path.join(json_path, json_file_name)
        # json_file_list = [json_file_name, "/home/vivek/Datasets/AmyB/amyb_wsi/XE19-010_1_AmyB_1_1.json"]
        # merge_json(json_file_list, "/home/vivek/Datasets/AmyB/amyb_wsi/test.json")
        # json_file_name = os.path.join(os.path.dirname(img), "XE19-010_1_AmyB_1_37894x_177901y_image.png[--series, 0].json")

        # json_file_name = "/home/vivek/Datasets/AmyB/amyb_wsi/test.json"
        
        print(json_file_name)
       
        if not exists(json_file_name):
            print("True")
            continue
        
        print("file name : ", json_file_name)

        with open(json_file_name) as f:
            data = json.load(f)
        
        for ele in tqdm(data):

            # Reset ids for each annotation
            ids = 1

            # Create an Empty mask of size similar to image
            id_mask = np.zeros(ID_MASK_SHAPE, dtype=np.uint8)

            region_id = 0
            prev_label = ""
            i = 0
            plaque_dict[ele['label']] = plaque_dict[ele['label']] + len(ele['region_attributes'])

            for region in ele['region_attributes']:

                # Get tileX and tileY
                tileX = region['tiles'][0]['tileId'][0]
                tileY = region['tiles'][0]['tileId'][1]
                tileWidth = region['tiles'][0]['tileBounds']["WH"][0]
                tileHeight = region['tiles'][0]['tileBounds']["WH"][1]

                # crop the image
                # get the bound-x and bounds-y, offset as Vips crops the empty spaces. Qupath does not
                tileX = (tileX * tileWidth)
                tileY = (tileY * tileHeight)

                vips_img_crop = vips_img.crop(tileX, tileY,tileWidth, tileHeight)
                print(tileX, tileY, tileWidth, tileHeight)
                # Region Bounds
                regX = region["roiBounds"]["XY"][0]
                regY = region["roiBounds"]["XY"][1]
                regWidth = region["roiBounds"]["WH"][0]
                regHeight = region["roiBounds"]["WH"][1]

                # region_crop = vips_img.crop(regX, regY, tileWidth, tileHeight)
                vips_img_crop = np.ndarray(buffer=vips_img_crop.write_to_memory(), dtype=np.uint8,
                                    shape=(vips_img_crop.height, vips_img_crop.width, vips_img_crop.bands))[..., :3]
                # region_img_crop = np.ndarray(buffer=vips_img_crop.write_to_memory(), dtype=np.uint8,
                #                 shape=(vips_img_crop.height, vips_img_crop.width, vips_img_crop.bands))[..., :3]

                # unpack from [x,y] to [x], [y]
                coords_x, coords_y = zip(*region['points'])

                coords_x = np.array(coords_x)
                coords_y = np.array(coords_y)

                x1 = tileX
                x2 = tileX + tileWidth 
                y1 = tileY 
                y2 = tileY + tileHeight


                # Remove overlap annotations
                if len(coords_x[coords_x > x2]) > 0 or len(coords_y[coords_y > y2]) > 0:
                    print('Overlap')
                    continue


                # Translate the coordinates to fit within the image crop
                coords_x = np.mod(coords_x, tileWidth)
                coords_y = np.mod(coords_y, tileHeight)



                # label
                label = ele['label']

                if i == 0:
                    ids = int(lablel2id[label])
                elif label == prev_label:
                    ids = int(lablel2id[label])

                # Use polygon2id function to create a mask
                id_mask = polygon2id(ID_MASK_SHAPE, id_mask, ids, coords_y, coords_x)

                # ids = ids + 5

                prev_label = label

                i+=1

                save_img(vips_img_crop, ele['filename'], tileX, tileY, image_save_dir, "image")
                save_img(id_mask, ele['filename'], tileX, tileY, mask_save_dir,"mask")
                id_mask = np.zeros(ID_MASK_SHAPE, dtype=np.uint8)

            if visualize:
                plt.imshow(id_mask)
                plt.show()

        print(plaque_dict)


def merge_json(json_files, json_output_file=None):
    """
    merge_json: a method to return the combined json of a list of json files

    """
    result = list()
    for f1 in json_files:
        with open(f1, 'r') as infile:
            result.extend(json.load(infile))

    with open(json_output_file, 'w') as output_file:
        json.dump(result, output_file)


In [7]:
dlb_wsi_dir = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD_Dataset/LBD/DLB_cases"
pdd_wsi_dir = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD_Dataset/LBD/PDD_cases/PDD_cases"
wsi_home_dir = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD_Dataset/LBD/"

In [8]:
json_path =  "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/LBD_Jsons_All_Intensity/"

In [9]:
f_list = os.listdir(json_path)
#f_list.remove('.DS_Store')

In [10]:
f_list

['PD130_Syn1_CG.json',
 'PD079_Syn1_CG.json',
 'PD092_Syn1_FCX.json',
 'PD113_Syn1_CG.json',
 '13_131_CG_aSyn_x200.json',
 '14_087_CG_aSyn_x200.json',
 '14_148_CG_aSyn_x200.json',
 'PD110_Syn1_CG.json',
 'PD034_Syn1_CG.json',
 'PD002_Syn1_CG.json',
 '15_005_CG_aSyn_x200.json',
 'PD067_Syn1_CG.json',
 '14_133_CG_aSyn_x200.json',
 'PD088_Syn1_CG.json',
 '14_075_CG_aSyn_x200.json',
 '12_060_CG_aSyn_x200.json',
 '14_053_CG_aSyn_x200.json',
 'PD041_Syn1_CG.json',
 '12_007_CG_aSyn_x200.json',
 '14_153_CG_aSyn_x200.json',
 'PD090_Syn1_CG.json',
 'PD133_Syn1_CG.json',
 '13_177_CG_aSyn_x200.json',
 '11_063_CG_aSyn_x200.json',
 'PD061_Syn1_CG.json',
 '14_036_CG_aSyn_x200.json',
 'PD013_Syn1_CG.json',
 'PD001_Syn1_CG.json',
 'PD753_aSyn_CG_x400.json',
 '14_073_CG_aSyn_x200.json',
 'PD131_Syn1_CG.json',
 '15_007_CG_aSyn_x200.json',
 'PD017_Syn1_CG.json']

In [11]:
import os
f_list_updated_names = []
for file in f_list:
    x = file.replace("SYN1", "Syn1")
    os.rename(os.path.join(json_path,file),os.path.join(json_path, x))
    f_list_updated_names.append(x)

In [12]:
f_list_updated_names

['PD130_Syn1_CG.json',
 'PD079_Syn1_CG.json',
 'PD092_Syn1_FCX.json',
 'PD113_Syn1_CG.json',
 '13_131_CG_aSyn_x200.json',
 '14_087_CG_aSyn_x200.json',
 '14_148_CG_aSyn_x200.json',
 'PD110_Syn1_CG.json',
 'PD034_Syn1_CG.json',
 'PD002_Syn1_CG.json',
 '15_005_CG_aSyn_x200.json',
 'PD067_Syn1_CG.json',
 '14_133_CG_aSyn_x200.json',
 'PD088_Syn1_CG.json',
 '14_075_CG_aSyn_x200.json',
 '12_060_CG_aSyn_x200.json',
 '14_053_CG_aSyn_x200.json',
 'PD041_Syn1_CG.json',
 '12_007_CG_aSyn_x200.json',
 '14_153_CG_aSyn_x200.json',
 'PD090_Syn1_CG.json',
 'PD133_Syn1_CG.json',
 '13_177_CG_aSyn_x200.json',
 '11_063_CG_aSyn_x200.json',
 'PD061_Syn1_CG.json',
 '14_036_CG_aSyn_x200.json',
 'PD013_Syn1_CG.json',
 'PD001_Syn1_CG.json',
 'PD753_aSyn_CG_x400.json',
 '14_073_CG_aSyn_x200.json',
 'PD131_Syn1_CG.json',
 '15_007_CG_aSyn_x200.json',
 'PD017_Syn1_CG.json']

In [13]:
#imagenames = sorted(glob.glob(os.path.join(wsi_home_dir, './*/*.svs')))
dlb_imagenames = sorted(glob.glob(os.path.join(dlb_wsi_dir, './*.svs')))
pdd_imagenames = sorted(glob.glob(os.path.join(pdd_wsi_dir, '././*.svs')))

In [15]:
REF_IMG_PATH = dlb_imagenames[0]
normalizer = normalization(REF_IMG_PATH)

Init Normalization


In [39]:
mask_save_dir = os.path.join(DATASET_PATH, "labels")
if not os.path.exists(mask_save_dir):
    os.makedirs(mask_save_dir)

# Image Folder
image_save_dir = os.path.join(DATASET_PATH, "images")
if not os.path.exists(image_save_dir):
    os.makedirs(image_save_dir)

In [16]:
plaque_dict = {'True': 0, 'Pre': 0, 'False': 0,'Unknown': 0,'White':0,'grey':0,'bg':0}

In [20]:
visualize = False
for geofile in f_list_updated_names:
    #geofile = "PD017_Syn1_CG.json"
    filename = geofile.replace(".json",".svs")
    geofile1 = os.path.join(json_path,geofile)
    with open(geofile1) as f:
        data = json.load(f)
    
    if filename.startswith("PD"):
        img = os.path.join(pdd_wsi_dir, filename)
    else:
        img = os.path.join(dlb_wsi_dir, filename)


    try:
        vips_img = Vips.Image.new_from_file(img, level=0)
    except:
        print("Error loading Vips Image", filename)
        continue

    mask_save_dir = os.path.join(DATASET_PATH,filename,"labels")
    if not os.path.exists(mask_save_dir):
        os.makedirs(mask_save_dir)

    # Image Folder
    image_save_dir = os.path.join(DATASET_PATH,filename, "images")
    if not os.path.exists(image_save_dir):
        os.makedirs(image_save_dir)

    #vips_img = normalizer.transform(vips_img)
    vinfo = get_vips_info(vips_img)
    for ele in tqdm(data):
        # Reset ids for each annotation
        ids = 1

        # Create an Empty mask of size similar to image
        id_mask = np.zeros(ID_MASK_SHAPE, dtype=np.uint8)

        region_id = 0
        prev_label = ""
        i = 0
        if ele['label'] in ["True","False","Pre"]:
            plaque_dict[ele['label'][0]] = plaque_dict[ele['label'][0]] + len(ele['region_attributes'])
        if len(ele['label'])==1:
            continue
        if ele['label'][1]=='3+' and ele['label'][0] in ["True","Pre","False"]:
            for region in ele['region_attributes']:
                # Get tileX and tileY
                tileX = region['tiles'][0]['tileId'][0]
                tileY = region['tiles'][0]['tileId'][1]
                tileWidth = region['tiles'][0]['tileBounds']["WH"][0]
                tileHeight = region['tiles'][0]['tileBounds']["WH"][1]

                # crop the image
                # get the bound-x and bounds-y, offset as Vips crops the empty spaces. Qupath does not
                tileX = (tileX * tileWidth)
                tileY = (tileY * tileHeight)

                vips_img_crop = vips_img.crop(tileX, tileY,tileWidth, tileHeight)
                #print(tileX, tileY, tileWidth, tileHeight)
                # Region Bounds
                regX = region["roiBounds"]["XY"][0]
                regY = region["roiBounds"]["XY"][1]
                regWidth = region["roiBounds"]["WH"][0]
                regHeight = region["roiBounds"]["WH"][1]

                # region_crop = vips_img.crop(regX, regY, tileWidth, tileHeight)
                vips_img_crop = np.ndarray(buffer=vips_img_crop.write_to_memory(), dtype=np.uint8,
                                    shape=(vips_img_crop.height, vips_img_crop.width, vips_img_crop.bands))[..., :3]
                # region_img_crop = np.ndarray(buffer=vips_img_crop.write_to_memory(), dtype=np.uint8,
                #                 shape=(vips_img_crop.height, vips_img_crop.width, vips_img_crop.bands))[..., :3]

                # unpack from [x,y] to [x], [y]
                coords_x, coords_y = zip(*region['points'])

                coords_x = np.array(coords_x)
                coords_y = np.array(coords_y)

                x1 = tileX
                x2 = tileX + tileWidth 
                y1 = tileY 
                y2 = tileY + tileHeight


                # Remove overlap annotations
                if len(coords_x[coords_x > x2]) > 0 or len(coords_y[coords_y > y2]) > 0:
                    print('Overlap')
                    continue


                # Translate the coordinates to fit within the image crop
                coords_x = np.mod(coords_x, tileWidth)
                coords_y = np.mod(coords_y, tileHeight)



                # label
                label = ele['label'][0]

                if i == 0:
                    ids = int(lablel2id[label])
                elif label == prev_label:
                    ids = int(lablel2id[label])

                # Use polygon2id function to create a mask
                id_mask = polygon2id(ID_MASK_SHAPE, id_mask, ids, coords_y, coords_x)

                # ids = ids + 5

                prev_label = label

                i+=1

                save_img(vips_img_crop, ele['filename'], tileX, tileY, image_save_dir, "image")
                save_img(id_mask, ele['filename'], tileX, tileY, mask_save_dir,"mask")
                id_mask = np.zeros(ID_MASK_SHAPE, dtype=np.uint8)

        if visualize:
            plt.imshow(id_mask)
            plt.show()

        print(plaque_dict)


    

{'': 'label, macro, thumbnail', 'AppMag': '20', 'DSR ID': '163.1.249.162', 'Date': '08/11/15', 'DisplayColor': '0', 'Exposure Scale': '0.000001', 'Exposure Time': '109', 'Filename': '24666', 'Focus Offset': '-0.000500', 'ICC Profile': 'ScanScope v1', 'ImageID': '24666', 'Left': '25.970015', 'LineAreaXOffset': '0.002305', 'LineAreaYOffset': '-0.000948', 'LineCameraSkew': '-0.001426', 'MPP': '0.5019', 'OriginalHeight': '28976', 'OriginalWidth': '38608', 'Parmset': 'Faintly Stained', 'ScanScope ID': 'SS1774', 'StripeWidth': '2032', 'Time': '17:29:20', 'Time Zone': 'GMT+01:00', 'Title': 'PD130-2', 'Top': '21.854683', 'User': '937e2d41-eed0-4110-b37f-1c142546c2c9', 'comment': 'Aperio Image Library v12.0.15 \r\n38608x28976 [0,100 37847x28876] (240x240) J2K/KDU Q=70|AppMag = 20|StripeWidth = 2032|ScanScope ID = SS1774|Filename = 24666|Title = PD130-2|Date = 08/11/15|Time = 17:29:20|Time Zone = GMT+01:00|User = 937e2d41-eed0-4110-b37f-1c142546c2c9|Parmset = Faintly Stained|MPP = 0.5019|Left = 

100%|██████████| 3/3 [00:00<00:00, 4641.43it/s]


{'True': 353, 'Pre': 136, 'False': 93, 'Unknown': 0, 'White': 53, 'grey': 70, 'bg': 30}
{'True': 353, 'Pre': 136, 'False': 93, 'Unknown': 0, 'White': 53, 'grey': 70, 'bg': 30}
{'True': 353, 'Pre': 136, 'False': 93, 'Unknown': 0, 'White': 53, 'grey': 70, 'bg': 30}
{'': 'label, macro, thumbnail', 'AppMag': '20', 'DSR ID': '163.1.249.162', 'Date': '05/26/15', 'DisplayColor': '0', 'Exposure Scale': '0.000001', 'Exposure Time': '109', 'Filename': '11607', 'Focus Offset': '-0.000500', 'ICC Profile': 'ScanScope v1', 'ImageID': '11607', 'Left': '19.360617', 'LineAreaXOffset': '0.002305', 'LineAreaYOffset': '-0.000948', 'LineCameraSkew': '-0.001426', 'MPP': '0.5019', 'OriginalHeight': '33044', 'OriginalWidth': '32512', 'ScanScope ID': 'SS1774', 'StripeWidth': '2032', 'Time': '20:54:07', 'Time Zone': 'GMT+01:00', 'Top': '21.016436', 'User': 'e90d5259-51c8-4da8-9092-b09564ebaf4b', 'comment': 'Aperio Image Library v12.0.15 \r\n32512x33044 [0,100 31871x32944] (240x240) J2K/KDU Q=70|AppMag = 20|Stri

  0%|          | 0/11 [00:00<?, ?it/s]

{'True': 353, 'Pre': 136, 'False': 93, 'Unknown': 0, 'White': 53, 'grey': 70, 'bg': 30}
{'True': 353, 'Pre': 136, 'False': 93, 'Unknown': 0, 'White': 53, 'grey': 70, 'bg': 30}
{'True': 353, 'Pre': 136, 'False': 93, 'Unknown': 0, 'White': 53, 'grey': 70, 'bg': 30}
{'True': 353, 'Pre': 136, 'False': 93, 'Unknown': 0, 'White': 53, 'grey': 70, 'bg': 30}
{'True': 353, 'Pre': 136, 'False': 93, 'Unknown': 0, 'White': 53, 'grey': 70, 'bg': 30}


...

{'True': 353, 'Pre': 136, 'False': 93, 'Unknown': 0, 'White': 53, 'grey': 70, 'bg': 30}


 14%|█▍        | 2/14 [00:35<03:08, 15.71s/it]

{'True': 353, 'Pre': 136, 'False': 93, 'Unknown': 0, 'White': 53, 'grey': 70, 'bg': 30}
{'True': 353, 'Pre': 136, 'False': 93, 'Unknown': 0, 'White': 53, 'grey': 70, 'bg': 30}
{'True': 353, 'Pre': 136, 'False': 93, 'Unknown': 0, 'White': 53, 'grey': 70, 'bg': 30}
{'True': 353, 'Pre': 136, 'False': 93, 'Unknown': 0, 'White': 53, 'grey': 70, 'bg': 30}


100%|██████████| 14/14 [00:39<00:00,  2.80s/it]

{'True': 353, 'Pre': 136, 'False': 93, 'Unknown': 0, 'White': 53, 'grey': 70, 'bg': 30}
{'True': 353, 'Pre': 136, 'False': 93, 'Unknown': 0, 'White': 53, 'grey': 70, 'bg': 30}
{'True': 353, 'Pre': 136, 'False': 93, 'Unknown': 0, 'White': 53, 'grey': 70, 'bg': 30}
{'True': 353, 'Pre': 136, 'False': 93, 'Unknown': 0, 'White': 53, 'grey': 70, 'bg': 30}
{'True': 353, 'Pre': 136, 'False': 93, 'Unknown': 0, 'White': 53, 'grey': 70, 'bg': 30}
{'True': 353, 'Pre': 136, 'False': 93, 'Unknown': 0, 'White': 53, 'grey': 70, 'bg': 30}
{'True': 353, 'Pre': 136, 'False': 93, 'Unknown': 0, 'White': 53, 'grey': 70, 'bg': 30}


In [58]:
input_path = '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/generated_data_v2/test/'

In [59]:
test_folders = glob.glob(os.path.join(input_path, "*"))

In [60]:
test_folders

['/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/generated_data_v2/test/labels',
 '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/generated_data_v2/test/images']